In [21]:
%pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
import requests
import pandas as pd
import polars as pl
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY not found in .env")

In [23]:
url = "https://api.511.org/transit/datafeeds"

In [24]:
import zipfile

params = {
    "api_key": API_KEY,
    "operator_id": "RG",
    "historic": "2025-03-so"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

with open("../data/march_2025_gtfs.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("../data/march_2025_gtfs.zip", "r") as zip_ref:
    zip_ref.extractall("march_2025_gtfs")

os.listdir("../data/march_2025_gtfs")

['transfers.txt',
 'agency.txt',
 'calendar_dates.txt',
 'stop_times.txt',
 'levels.txt',
 'directions.txt',
 'trips.txt',
 'stop_observations.txt',
 'pathways.txt',
 'stops.txt',
 'routes.txt']

In [25]:
obs = pd.read_csv(
    "../data/march_2025_gtfs/stop_observations.txt",
    dtype={
        "trip_id": "string",
        "vehicle_id": "string",
        "route_id": "string",
        "agency_id": "string",
        "from_stop_id": "string",
        "to_stop_id": "string"
    }
)

sf = obs[
    obs["agency_id"].str.contains("SF", na=False)
].copy()

sf.head(10)

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,scheduled_dwell_time_secs,route_id,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time
416,SF:11721738_M31:20250331,NaN,0,20250331,8857,1,05:00:00,05:00:00,0,0.0,0.0,SF:29,SF,1.0,<NA>,14648,05:00:00,05:00:00
417,SF:11721738_M31:20250331,NaN,0,20250331,8857,2,05:01:01,05:01:01,0,0.0,0.0,SF:29,SF,1.0,14648,14647,05:00:33,05:00:33
418,SF:11721738_M31:20250331,NaN,0,20250331,8857,3,05:01:58,05:01:58,0,0.0,0.0,SF:29,SF,1.0,14647,14646,05:01:14,05:01:14
419,SF:11721738_M31:20250331,NaN,0,20250331,8857,4,05:02:22,05:02:22,0,0.0,0.0,SF:29,SF,1.0,14646,14645,05:01:59,05:01:59
420,SF:11721738_M31:20250331,NaN,0,20250331,8857,5,05:03:38,05:03:38,0,0.0,0.0,SF:29,SF,1.0,14645,14784,05:02:54,05:02:54
421,SF:11721738_M31:20250331,NaN,0,20250331,8857,6,05:03:57,05:03:57,0,0.0,0.0,SF:29,SF,1.0,14784,15955,05:03:27,05:03:27
422,SF:11721738_M31:20250331,NaN,0,20250331,8857,7,05:04:32,05:04:32,0,0.0,0.0,SF:29,SF,1.0,15955,15115,05:04:27,05:04:27
423,SF:11721738_M31:20250331,NaN,0,20250331,8857,8,05:05:09,05:05:09,0,0.0,0.0,SF:29,SF,1.0,15115,15116,05:05:07,05:05:07
424,SF:11721738_M31:20250331,NaN,0,20250331,8857,9,05:05:52,05:05:52,0,0.0,0.0,SF:29,SF,1.0,15116,14976,05:05:43,05:05:43
425,SF:11721738_M31:20250331,NaN,0,20250331,8857,10,05:06:25,05:06:25,0,0.0,0.0,SF:29,SF,1.0,14976,14785,05:06:23,05:06:23


In [26]:
sf.shape

(8160083, 18)

In [18]:
sf.isna().sum().sort_values(ascending=False)

trip_start_time              8160083
from_stop_id                  220269
scheduled_departure_time         319
scheduled_arrival_time           319
scheduled_dwell_time_secs        319
direction_id                      35
to_stop_id                         0
agency_id                          0
route_id                           0
trip_id                            0
uncertainty                        0
observed_departure_time            0
observed_arrival_time              0
stop_sequence                      0
vehicle_id                         0
service_date                       0
schedule_relationship              0
dwell_time_secs                    0
dtype: int64

In [19]:
sf.describe(include="all")

,trip_id,trip_start_time,schedule_relationship,service_date,vehicle_id,stop_sequence,observed_arrival_time,observed_departure_time,uncertainty,dwell_time_secs,scheduled_dwell_time_secs,route_id,agency_id,direction_id,from_stop_id,to_stop_id,scheduled_arrival_time,scheduled_departure_time
count,8160083,0,8160083.0,8.160083e+06,8160083,8.160083e+06,8160083,8160083,8160083.0,8160083.0,8159764.0,8160083,8160083,8.160048e+06,7939814,8160083,8159764,8159764
unique,62013,0,NaN,NaN,1050,NaN,96273,96273,NaN,NaN,NaN,70,1,NaN,3263,3276,79928,79928
top,SF:11733360_M31:20250314,NaN,NaN,NaN,8526,NaN,06:00:00,06:00:00,NaN,NaN,NaN,SF:29,SF,NaN,14725,15926,07:00:00,07:00:00
freq,1207,NaN,NaN,NaN,19027,NaN,779,779,NaN,NaN,NaN,392786,8160083,NaN,11356,13552,2636,2636
mean,NaN,NaN,0.0,2.025032e+07,NaN,2.356641e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,5.059219e-01,NaN,NaN,NaN,NaN
std,NaN,NaN,0.0,7.754517e+00,NaN,1.732639e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,4.999650e-01,NaN,NaN,NaN,NaN
min,NaN,NaN,0.0,2.025030e+07,NaN,1.000000e+00,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN
25%,NaN,NaN,0.0,2.025031e+07,NaN,1.000000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN
50%,NaN,NaN,0.0,2.025032e+07,NaN,2.000000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,1.000000e+00,NaN,NaN,NaN,NaN
75%,NaN,NaN,0.0,2.025032e+07,NaN,3.300000e+01,NaN,NaN,0.0,0.0,0.0,NaN,NaN,1.000000e+00,NaN,NaN,NaN,NaN


In [20]:
sf["route_id"].nunique()

70